Setup and Data Loading


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv("EVCarsSales.csv")
df.head(10)

,region,category,parameter,mode,powertrain,year,unit,value
0,Australia,Historical,EV sales,Cars,BEV,2011,Vehicles,49.00000
1,Australia,Historical,EV stock share,Cars,EV,2011,percent,0.00039
2,Australia,Historical,EV sales share,Cars,EV,2011,percent,0.00650
3,Australia,Historical,EV stock,Cars,BEV,2011,Vehicles,49.00000
4,Australia,Historical,EV stock,Cars,BEV,2012,Vehicles,220.00000
5,Australia,Historical,EV stock,Cars,PHEV,2012,Vehicles,80.00000
6,Australia,Historical,EV sales,Cars,PHEV,2012,Vehicles,80.00000
7,Australia,Historical,EV sales share,Cars,EV,2012,percent,0.03000
8,Australia,Historical,EV stock share,Cars,EV,2012,percent,0.00240
9,Australia,Historical,EV sales,Cars,BEV,2012,Vehicles,170.00000


- **Task 1**
Identify data quality issues in the dataset.


In [ ]:
# Check for missing values and duplicates
print("Missing Values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

# logical Issue: Unit inconsistency
print("\nUnique Units:", df['unit'].unique())

Missing Values:
 region        0
category      0
parameter     0
mode          0
powertrain    0
year          0
unit          0
value         0
dtype: int64

Duplicates: 0

Unique Units: ['Vehicles' 'percent' 'GWh' 'Milion barrels per day'
 'Oil displacement, million lge']



- **Task 2**
Apply one missing value strategy and explain why.

In [ ]:
# Even if there are no NaNs in this sample, this is the robust way to handle them:
df['value'] = df['value'].fillna(df['value'].median())

# Explanation: We use Median instead of Mean because the data is skewed by 
# small percentages vs large vehicle counts. The median is less sensitive to these gaps.


- **Task 3**
Detect and handle outliers using IQR.

In [4]:
Q1 = df['value'].quantile(0.25)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = df[(df['value'] < lower_bound) | (df['value'] > upper_bound)]
print("Outliers detected:\n", outliers)

# Handle outliers by capping (Winsorization)
df['value_cleaned'] = np.clip(df['value'], lower_bound, upper_bound)

Outliers detected:
          region    category                      parameter  mode powertrain  \
53    Australia  Historical                       EV stock  Cars        BEV   
63    Australia  Historical                       EV stock  Cars        BEV   
64    Australia  Historical                       EV sales  Cars        BEV   
67    Australia  Historical                       EV sales  Cars        BEV   
68    Australia  Historical                       EV stock  Cars        BEV   
...         ...         ...                            ...   ...        ...   
3790      World  Historical  Oil displacement, million lge  Cars         EV   
3791      World  Historical                       EV stock  Cars       FCEV   
3793      World  Historical                       EV stock  Cars        BEV   
3794      World  Historical                       EV sales  Cars        BEV   
3797      World  Historical             Electricity demand  Cars         EV   

      year                     


- **Task 4**
Normalize numerical features using both Min-Max and Z-score.

In [5]:
# Min-Max Scaling (0 to 1)
min_max = MinMaxScaler()
df['value_minmax'] = min_max.fit_transform(df[['value']])

# Z-score Normalization (Mean=0, Std=1)
z_score = StandardScaler()
df['value_zscore'] = z_score.fit_transform(df[['value']])

df[['value', 'value_minmax', 'value_zscore']]

,value,value_minmax,value_zscore
0,4.900000e+01,1.749999e-06,-0.123306
1,3.900000e-04,1.339286e-11,-0.123366
2,6.500000e-03,2.316071e-10,-0.123366
3,4.900000e+01,1.749999e-06,-0.123306
4,2.200000e+02,7.857142e-06,-0.123097
...,...,...,...
3793,2.800000e+07,1.000000e+00,34.092555
3794,9.500000e+06,3.392857e-01,11.485607
3795,1.800000e+01,6.428566e-07,-0.123344
3796,3.200000e+00,1.142852e-07,-0.123362



- **Task 5**
Apply PCA and interpret explained variance.

In [6]:
# 1. Prepare data (One-Hot Encoding categories)
df_numeric = pd.get_dummies(df.drop(columns=['value_minmax', 'value_zscore']))

# 2. Apply PCA
pca = PCA(n_components=2)
pca_results = pca.fit_transform(df_numeric)

# 3. Interpretation
exp_var = pca.explained_variance_ratio_
print(f"Explained Variance: PC1 = {exp_var[0]:.2%}, PC2 = {exp_var[1]:.2%}")

Explained Variance: PC1 = 99.99%, PC2 = 0.01%
